# Notebook 01b: Autoencoders, VAEs, and Generative Sampling

### Overview
In this notebook, we explore **Autoencoders (AEs)** and **Variational Autoencoders (VAEs)** for representation learning and generative sampling of cell micrographs.

**Key Learning Objectives**:
1. Train a Convolutional Autoencoder (AE) vs a Variational Autoencoder (VAE).
2. Understand the ELBO loss function ($	ext{Reconstruction} + eta 	ext{KL}$).
3. Perform **Generative Sampling** by drawing $z \sim \mathcal{N}(0, I)$ to synthesize new cell images from scratch.
4. Perform **Latent Space Interpolation** between a GFP-negative cell ($z_A$) and a GFP-positive cell ($z_B$).

In [ ]:
# ==========================================================
# 1. Environment Setup & Data Loading
# ==========================================================
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

if 'google.colab' in sys.modules:
    print("[+] Google Colab detected!")
    !git clone https://github.com/ayakimovich/GenAI4BIA.git /content/GenAI4BIA
    %cd /content/GenAI4BIA/practical
    !pip install -r ../requirements.txt -q
    sys.path.append(os.path.abspath("."))
else:
    sys.path.append(os.path.abspath("."))

from src.data import VIRVSDataset
from src.models import Autoencoder, ConvVAE
from src.generate_mock_virvs_data import create_dataset_directory

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Using device: {device}")

data_dir = "./data/mock_virvs"
create_dataset_directory(data_dir, num_train=30, num_val=10, image_size=(256, 256))

train_dataset = VIRVSDataset(root_dir=data_dir, split="train", normalize_range=(0.0, 1.0))
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

## 2. Train Variational Autoencoder (VAE)
The VAE maps fluorescence cell micrographs $x$ to latent distribution $(\mu, \sigma)$ and optimizes the ELBO objective.

In [ ]:
vae = ConvVAE(in_channels=1, latent_dim=32).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)

def vae_loss(x_recon, x, mu, log_var, beta=1.0):
    recon_loss = nn.functional.mse_loss(x_recon, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return recon_loss + beta * kl_loss, recon_loss, kl_loss

num_epochs = 10
print("[+] Training Variational Autoencoder (VAE)...")
for epoch in range(1, num_epochs + 1):
    vae.train()
    total_loss_accum = 0.0
    for batch in train_loader:
        x = batch["fluorescence"].to(device)
        optimizer.zero_grad()
        x_recon, mu, log_var = vae(x)
        loss, r_loss, k_loss = vae_loss(x_recon, x, mu, log_var, beta=1.0)
        loss.backward()
        optimizer.step()
        total_loss_accum += loss.item()
        
    if epoch % 2 == 0 or epoch == num_epochs:
        print(f"Epoch [{epoch:02d}/{num_epochs:02d}] - Total Loss: {total_loss_accum / len(train_dataset):.2f}")

## 3. Generative Sampling: $z \sim \mathcal{N}(0, I)$
Because the KL divergence regularized the latent space against a standard Gaussian prior $\mathcal{N}(0, I)$, we can synthesize brand new cell images by sampling random $z$ vectors!

In [ ]:
# Draw 8 random latent samples from prior N(0, I)
with torch.no_grad():
    synthetic_cells = vae.sample(num_samples=8, device=device)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    cell_img = synthetic_cells[i, 0].cpu().numpy()
    ax.imshow(cell_img, cmap="magma")
    ax.set_title(f"Sample {i+1}", fontsize=10)
    ax.axis("off")

plt.suptitle("Generative Sampling from VAE Prior: z ~ N(0, I)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Latent Space Interpolation: GFP-Negative to GFP-Positive
We pick two latent vectors ($z_A$ representing a low-expression GFP cell and $z_B$ representing a high-expression GFP cell) and linearly interpolate $z(lpha) = (1-lpha) z_A + lpha z_B$.

In [ ]:
# Pick two samples from validation batch
batch = next(iter(train_loader))
x_batch = batch["fluorescence"].to(device)

with torch.no_grad():
    mu, _ = vae.encode(x_batch)
    zA = mu[0:1] # Cell A latent vector
    zB = mu[1:2] # Cell B latent vector

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(alphas), figsize=(12, 3))

with torch.no_grad():
    for idx, alpha in enumerate(alphas):
        z_interp = (1.0 - alpha) * zA + alpha * zB
        cell_interp = vae.decode(z_interp)[0, 0].cpu().numpy()
        axes[idx].imshow(cell_interp, cmap="magma")
        axes[idx].set_title(f"α = {alpha:.2f}", fontsize=10, fontweight="bold")
        axes[idx].axis("off")

plt.suptitle("Latent Space Interpolation (Cell Morphing)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()